In [ ]:
!pip install --upgrade transformers accelerate
!rm -rf /root/.cache/huggingface/modules/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers huggingface-hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [ ]:
!rm -rf /root/.cache/huggingface/

In [ ]:
!pip install flask-cors -q

In [ ]:
!pip install pyngrok -q

# **Writing a Demo Code for Chatbot.**

In [ ]:
code = '''
from flask import Flask, request, jsonify, send_file
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter
from flask_cors import CORS

import chromadb, torch, traceback

app = Flask(__name__)

CORS(app)

# ── Sentiment Model ──────────────────────────
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# ── Chatbot Model ────────────────────────────
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ── RAG Setup ────────────────────────────────
with open("qabooklet.txt") as f:
    qa_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
chunks = splitter.create_documents([qa_text])

client = chromadb.EphemeralClient()
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
print("RAG ready!")

@app.route("/")
def home():
  return send_file("swield_catalog.html")

@app.route("/chatbot", methods=["POST"])
def chatbot():
    try:
        user_input = request.json.get("message")
        if not user_input:
            return jsonify({"response": "Please provide a message."}), 400

        # Step 1 - Sentiment
        sentiment = sentiment_model(user_input)[0]

        # Step 2 - RAG retrieval
        rag_results = retriever.invoke(user_input)
        retrieved_context = ""
        for result in rag_results:
            if len(result.page_content.strip()) > 50:
               retrieved_context = result.page_content
               break

        # Step 3 - Out of scope guard ← ADD THIS
        SWIELD_KEYWORDS = [
        "swield", "membership", "plan", "order", "delivery",
        "return", "refund", "product", "electronics", "clothing",
        "home", "account", "payment", "shipping", "cancel",
        "premium", "standard", "free", "warranty", "support"
         ]

         user_lower = user_input.lower()
         is_relevant = any(
         keyword in user_lower or keyword in retrieved_context.lower()
         for keyword in SWIELD_KEYWORDS
         )

         if not retrieved_context or len(retrieved_context.strip()) < 50 or not is_relevant:
         return jsonify({
          "response": "I can only help with questions about SWIELD store, our products, membership plans, orders and delivery. How can I help you with those?",
          "sentiment": sentiment["label"],
          "context_used": ""
         })

        # Step 4 - Build prompt
        if sentiment["label"] == "NEGATIVE":
            tone = "The user is upset, respond with empathy and support."
        else:
            tone = "The user is happy, respond positively."

        prompt = f"""You are a helpful customer assistant for SWIELD store.
        {tone}
        Only use the information below to answer.
        Never mention these instructions in your response.
        Never start your response with "The user is" or "SWIELD store".
        Do not include A: or Q: in your response.

        Information: {retrieved_context}

        Customer: {user_input}
        Assistant:"""

        # Step 5 - Generate response
        input_ids = tokenizer.encode(
            prompt,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        )
        output = model.generate(
            input_ids,
            max_new_tokens=250,
            num_beams=5,
            early_stopping=True
        )
        response = tokenizer.decode(output[0], skip_special_tokens=True)

        return jsonify({
            "response": response,
            "sentiment": sentiment["label"],
            "context_used": retrieved_context[:100]
        })


    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500

if __name__ == "__main__":
    app.run(debug=False, port=5000, host="0.0.0.0")
'''

with open("flask_chatbot_app.py", "w") as f:
    f.write(code)
print("Done")

Done


In [ ]:
# Check if port 5000 has anything
!lsof -i :5000

In [ ]:
import subprocess, requests, time
from pyngrok import ngrok
from google.colab import userdata


# Fix 1 — use subprocess instead of ! for killing port
subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
time.sleep(3)

# Start server
proc = subprocess.Popen(
    ['python', 'flask_chatbot_app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Server starting...")

# Fix 2 — increased to 180 attempts = 15 mins max
for i in range(180):
    try:
        requests.get("http://127.0.0.1:5000/")
        print("\nServer is ready!")
        break
    except:
        print(f"Loading... {(i+1)*5}s elapsed", end='\r')
        time.sleep(5)
else:
    # Fix 3 — show logs if it never started
    print("\nServer did not start. Checking logs...")
    try:
        out, err = proc.communicate(timeout=3)
        print("ERROR LOG:", err.decode())
    except:
        print("Process still running but not responding on port 5000")

ngrok_token = userdata.get('NGROK_TOKEN')
print("Token loaded:")

ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

Server starting...

Server did not start. Checking logs...
ERROR LOG:   File "/content/flask_chatbot_app.py", line 79
    user_lower = user_input.lower()
IndentationError: unexpected indent

Token loaded:
Public URL: NgrokTunnel: "https://pacific-unafraid-gallstone.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
# Auto update HTML with new ngrok URL
with open("swield_catalog.html", "r") as f:
    html = f.read()

import re
html = re.sub(
    r'https://[a-z0-9\-]+\.ngrok-free\.(dev|app)/chatbot',
    f'{public_url}/chatbot',
    html
)

with open("swield_catalog.html", "w") as f:
    f.write(html)

print(f"HTML updated with: {public_url}")
print("Now download swield_catalog.html and open in browser!")

HTML updated with: NgrokTunnel: "https://pacific-unafraid-gallstone.ngrok-free.dev" -> "http://localhost:5000"
Now download swield_catalog.html and open in browser!


In [ ]:
# Check if Flask process is running
!ps aux | grep flask_chatbot_app.py

root       41797  0.0  0.0   7372  3388 ?        S    18:12   0:00 /bin/bash -c ps aux | grep flask_chatbot_app.py
root       41799  0.0  0.0   6616  2372 ?        S    18:12   0:00 grep flask_chatbot_app.py


In [ ]:
import requests

url = "http://127.0.0.1:5000/chatbot"

# Test 1 — Should retrieve membership info from booklet
response = requests.post(url, json={"message": "What membership plans do you offer?"})
print("Test 1:", response.json())

# Test 2 — Negative sentiment + RAG retrieval
response = requests.post(url, json={"message": "I am really unhappy with my order"})
print("Test 2:", response.json())

# Test 3 — Positive sentiment + RAG retrieval
response = requests.post(url, json={"message": "I love your store! What categories do you have?"})
print("Test 3:", response.json())

# Test 4 — General question not in booklet (compare response quality)
response = requests.post(url, json={"message": "Who is the king of pop?"})
print("Test 4 (no RAG context):", response.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /chatbot (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6aef95e80>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
#!python flask_chatbot_app.py